> ### 이 노트북 읽는 법
> 이 파일은 `RecSys_GCF_sol.ipynb`에 **파이썬 초보자용 해설**을 덧붙인 버전입니다.
> - 원본의 셀(설명·코드·실행 결과)은 **그대로** 두었습니다.
> - 번호가 붙은 코드 셀 **바로 아래**마다 `📘 셀 N 해설` 칸을 추가했습니다. 맨 앞 **Setup 부분(셀 1 ~ 3)은 해설을 생략**했고, 셀 4 ~ 14를 설명합니다.
> - 해설은 **한 줄 요약 → 코드 뜯어보기 → 출력 해석** 순서입니다. 함수를 정의만 하는 셀은 출력이 없으므로 코드 설명 위주입니다.
>
> **이 노트북의 큰 흐름**
> 1. 영화 평점 데이터(MovieLens)를 내려받는다 → 2. "사용자가 영화에 평점을 줬다"를 **선(edge)** 으로 하는 **그래프**를 만든다 → 3. 그래프 위에서 이웃의 정보를 모으는 **NGCF** 모델을 만든다 → 4. "본 영화가 안 본 영화보다 점수가 높게" 나오도록 학습한다 → 5. 사용자마다 **추천 10개**를 뽑아 얼마나 맞는지 평가한다.
>
> **Setup에서 만들어 두고 뒤에서 계속 쓰는 것들**
> - `torch`, `nn`, `F` : PyTorch와 그 안의 신경망 도구들 (`F`는 `torch.nn.functional`)
> - `np`, `pd` : numpy, pandas / `train_test_split` : 데이터를 나누는 함수
> - `download_url`, `extract_zip` : 파일을 내려받고 압축을 푸는 함수 (`torch_geometric` 제공)
> - `math`, `defaultdict` : 수학 함수 / 기본값이 있는 사전(dictionary)
> - `device` : 계산을 실행할 장치 (GPU가 있으면 `cuda`)
> - 셀 1의 `format_pytorch_version`, `format_cuda_version` 등은 설치 과정에서 쓰던 코드가 남은 것으로, 뒤에서 사용하지 않습니다.
>
> **미리 알아 둘 용어**
> - **추천 시스템** : 사용자가 좋아할 만한 항목(여기서는 영화)을 골라 주는 시스템
> - **협업 필터링(Collaborative Filtering, CF)** : "취향이 비슷한 사람들은 비슷한 것을 좋아한다"는 생각으로, 사용자들의 **행동 기록만으로** 추천하는 방법
> - **그래프** : **점(node)** 과 그것을 잇는 **선(edge)** 으로 이루어진 구조. 여기서는 사용자와 영화가 점이고, 평점을 준 관계가 선입니다.
> - **임베딩(embedding)** : 사용자나 영화 하나를 **숫자 여러 개(벡터)** 로 나타낸 것. 학습을 통해 취향이 비슷하면 벡터도 비슷해집니다.
> - **GNN(그래프 신경망)** : 각 점이 **이웃 점들의 정보를 모아** 자기 벡터를 갱신하는 신경망
> - **NGCF(Neural Graph Collaborative Filtering)** : GNN을 협업 필터링에 적용한 모델
> - **implicit feedback(암묵적 피드백)** : "몇 점을 줬는가"가 아니라 **"봤는가/안 봤는가"** 만 사용하는 데이터

# Setup

In [1]:
# 1
import torch
import torch.nn.functional as F
from torch import nn

def format_pytorch_version(version):
  return version.split('+')[0]

TORCH_version = torch.__version__
TORCH = format_pytorch_version(TORCH_version)

def format_cuda_version(version):
  return 'cu' + version.replace('.', '')

CUDA_version = torch.version.cuda
CUDA = format_cuda_version(CUDA_version)

!pip install torch-geometric

In [2]:
# 2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import random
from tqdm.notebook import tqdm
from torch_geometric.data import download_url, extract_zip
import math
from collections import defaultdict

In [3]:
# 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data Preprocessing

In [4]:
# 4
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
extract_zip(download_url(url, '.'), '.')

ratings_path = './ml-latest-small/ratings.csv'
rating_df = pd.read_csv(ratings_path)
print(len(rating_df))
print(rating_df['userId'].nunique())
print(rating_df['movieId'].nunique())

100836
610
9724


Extracting .\ml-latest-small.zip


### 📘 셀 4 해설
**한 줄 요약:** **MovieLens** 영화 평점 데이터를 내려받아 표(`rating_df`)로 읽습니다.

- `extract_zip(download_url(url, '.'), '.')` : 안쪽부터 읽습니다. `download_url(url, '.')`로 현재 폴더(`.`)에 zip 파일을 내려받고, `extract_zip`으로 압축을 풉니다.
- `pd.read_csv(ratings_path)` : csv 파일을 표로 읽습니다. 열은 `userId`, `movieId`, `rating`, `timestamp`입니다.
- `len(rating_df)` : 표의 줄 수 / `.nunique()` : **서로 다른 값의 개수**

**출력 해석**
- `100836` : 평점 기록 약 10만 건
- `610` : 사용자 610명 / `9724` : 영화 9,724편
- 모든 사용자가 모든 영화를 봤다면 610 × 9,724 ≈ 593만 건이 있어야 하는데 실제로는 약 10만 건(약 1.7%)뿐입니다. 이렇게 **대부분이 비어 있는 것**이 추천 데이터의 특징이며, 비어 있는 칸 중에서 좋아할 만한 것을 찾는 것이 추천의 목표입니다.

In [5]:
# 5
# Transform UserID and MovieID into sequential indices
user_encoder = {user: idx for idx, user in enumerate(rating_df['userId'].unique())}
movie_encoder = {movie: idx for idx, movie in enumerate(rating_df['movieId'].unique())}

rating_df['userId'] = rating_df['userId'].map(user_encoder)
rating_df['movieId'] = rating_df['movieId'].map(movie_encoder)

num_users = len(user_encoder)
num_movies = len(movie_encoder)

### 📘 셀 5 해설
**한 줄 요약:** 사용자 번호와 영화 번호를 **0부터 시작하는 연속된 번호**로 바꿉니다.

원래 `movieId`는 1, 3, 6, 47, …처럼 중간이 비어 있고 최댓값이 19만이 넘습니다. 뒤에서 임베딩 표의 "몇 번째 줄"로 쓰려면 0, 1, 2, …처럼 **빈틈없는 번호**여야 합니다.

- `{user: idx for idx, user in enumerate(rating_df['userId'].unique())}` : **사전(dictionary)을 한 줄로 만드는 문법**입니다.
  - `rating_df['userId'].unique()` : 서로 다른 사용자 번호를 등장 순서대로 나열
  - `enumerate(...)` : 하나씩 꺼내면서 0, 1, 2, … 번호를 함께 줌
  - 결과: `{원래 번호: 새 번호}` 형태의 사전
- `rating_df['userId'].map(user_encoder)` : 열의 모든 값을 사전에 따라 새 번호로 바꿉니다.
- `num_users = len(user_encoder)` : 사용자 수(610) / `num_movies` : 영화 수(9,724)

In [6]:
# 6
# Set seed for better replication
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(2025)

### 📘 셀 6 해설
**한 줄 요약:** **난수를 고정**하는 함수를 만들고 실행합니다.

- 가중치의 초기값, 데이터 나누기, 무작위 샘플 뽑기 등이 모두 난수를 사용합니다. 시작점(seed)을 고정해야 **다시 실행해도 비슷한 결과**가 나옵니다.
- `torch`, `torch.cuda`(GPU), `numpy`, 파이썬 기본 `random`이 각자 난수를 만들기 때문에 **네 군데를 모두** 고정합니다.
- `torch.backends.cudnn.deterministic = True`, `benchmark = False` : GPU 계산도 매번 같은 방식으로 하게 하는 설정입니다.
- `set_seed(2025)` : 함수를 실행합니다. 2025라는 숫자 자체에는 의미가 없습니다.

# Create Graph from Data

In [7]:
# 7
# Create Edge for Graph
# We generate edge between user and movie when user rates movie higher than (or equal to) 1
# Adjacency matrix 대신 edge_index를 넣어서도 진행가능하다.
def create_edge_index(df, rating_threshold=1.0):
    src, dst = [], []
    for _, row in df.iterrows():
        if row['rating'] >= rating_threshold:
            src.append(row['userId'])
            # item indices after user indices
            dst.append(row['movieId'] + num_users)
    return torch.tensor([src, dst], dtype=torch.long)

edge_index = create_edge_index(rating_df)
print(edge_index)

tensor([[   0,    0,    0,  ...,  609,  609,  609],
        [ 610,  611,  612,  ..., 3731, 2002, 3483]])


### 📘 셀 7 해설
**한 줄 요약:** 평점 기록을 **그래프의 선(edge) 목록**으로 바꿉니다.

**그래프를 표현하는 방법** — 점이 1만 개가 넘으므로 "모든 점 × 모든 점" 크기의 표(adjacency matrix)를 만들면 대부분이 0인 거대한 표가 됩니다. 대신 **실제로 이어진 선만** 적어 둡니다. 이것이 `edge_index`입니다.

```
edge_index = [[선의 출발점들 ...],    ← 사용자 번호
              [선의 도착점들 ...]]    ← 영화 번호
```

- `for _, row in df.iterrows():` : 표를 한 줄씩 꺼냅니다. 줄 번호는 필요 없어 `_`로 받습니다.
- `if row['rating'] >= rating_threshold:` : 평점이 1.0 이상이면 선을 만듭니다. 평점은 0.5 ~ 5.0이므로 **0.5점만 제외**됩니다.
- `src.append(row['userId'])` : 출발점 = 사용자 번호
- `dst.append(row['movieId'] + num_users)` : 도착점 = **영화 번호 + 610**

**왜 610을 더할까?** 사용자도 0번부터, 영화도 0번부터 시작하므로 그대로 쓰면 번호가 겹칩니다. 영화 번호를 뒤로 밀어 **하나의 번호 체계**로 만듭니다.

| 점의 번호 | 대상 |
|---|---|
| 0 ~ 609 | 사용자 610명 |
| 610 ~ 10333 | 영화 9,724편 |

여기서 **평점이 몇 점인지는 버리고 "봤다"는 사실만** 남깁니다. 이것이 implicit feedback입니다.

**출력 해석**
- 첫째 줄 `[0, 0, 0, ..., 609, 609, 609]` : 출발점(사용자). 0번 사용자의 기록부터 609번 사용자의 기록까지 순서대로 들어 있습니다.
- 둘째 줄 `[610, 611, 612, ..., 3731, 2002, 3483]` : 도착점(영화). 모두 610 이상입니다.
- 가운데 `...`은 너무 길어 생략했다는 표시입니다. 선은 약 9만 9천 개입니다.

In [8]:
# 8
# Split indices into train/val/test set (label for train/val/test set)
train_indices, test_indices = train_test_split(range(edge_index.size(1)), test_size=0.2)
val_indices, test_indices = train_test_split(test_indices, test_size=0.5)

train_edge_index = edge_index[:, train_indices]
val_edge_index = edge_index[:, val_indices]
test_edge_index = edge_index[:, test_indices]

### 📘 셀 8 해설
**한 줄 요약:** 선(edge)들을 **학습 80% / 검증 10% / 테스트 10%** 로 나눕니다.

- `range(edge_index.size(1))` : 선의 번호 0, 1, 2, … (`size(1)`은 선의 개수)
- 첫 번째 `train_test_split(..., test_size=0.2)` : 80%와 20%로 나눔
- 두 번째 `train_test_split(test_indices, test_size=0.5)` : 그 20%를 다시 **반반**으로 나눠 검증 10%, 테스트 10%
- `edge_index[:, train_indices]` : "모든 행(출발점·도착점), 고른 번호의 열"만 꺼냅니다.

**세 묶음의 역할**

| 묶음 | 역할 |
|---|---|
| 학습(train) | 모델이 배우는 데 사용. **그래프의 선으로도 사용** |
| 검증(validation) | 학습 중간중간 성능을 확인 |
| 테스트(test) | 학습이 끝난 뒤 최종 성능을 확인 |

검증·테스트의 선은 학습 때 **숨겨 둔 정답**입니다. "이 사용자가 실제로 본 영화인데 모델에게는 알려 주지 않은 것"을 모델이 추천해 내는지 확인합니다.

# Model: NGCF


### Model Structure of NGCF

<table>
  <tr>
    <td style="text-align: center;">
      <img src="https://drive.google.com/uc?id=1J0FBPjW8IhYF87hIkuLEKH6gKIlZaRI3" height="200" width="300">
      <br>
      <span>NGCF Model</span>
    </td>
  </tr>
</table>

In [9]:
# 9
### TODO : NGCF Layer 완성 ###

class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        # self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        '''
        edge_index : 엣지 정보 (src, dst)의 집합.
        node_features: node별 이전 layer에서 생성된 벡터 정보가 담긴 matrix (H^(l-1)) (|V| * d)
        '''

        src, dst = edge_index # src : user, dst : movie

        # calculate node degree
        deg = torch.zeros(node_features.size(0), device=node_features.device)
        # calculate user degree
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        # calculate movie degree
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        # calculate 1/(root(deg(u)) * root(deg(i))) for all edge
        norm = 1.0/torch.sqrt(deg[src]*deg[dst])

        src_feat = node_features[src] # H_u
        dst_feat = node_features[dst] # H_i

        edge_messages_for_src = self.W1(dst_feat) + self.W2(dst_feat*src_feat)
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = self.W1(src_feat) + self.W2(src_feat*dst_feat)
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like(node_features)
        aggregated_messages.index_add(0, src, edge_messages_for_src)
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])

        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        aggregated_features = self.leaky_relu(aggregated_messages)

        # engineering approach
        # aggregated_features = self.dropout(aggregated_features)
        return aggregated_features

### 📘 셀 9 해설
**한 줄 요약:** [실습 정답] NGCF의 **층(layer) 하나**를 만듭니다. 이 층은 **각 점이 이웃 점들에게서 메시지를 받아 자기 벡터를 새로 만드는** 일을 합니다.

**핵심 아이디어** — 사용자의 벡터에 "그 사람이 본 영화들"의 정보를 섞고, 영화의 벡터에 "그 영화를 본 사용자들"의 정보를 섞습니다. 그러면 같은 영화를 본 사용자끼리 벡터가 비슷해집니다.

**부품 (`__init__`)**
- `self.W1`, `self.W2` : 벡터를 변환하는 Linear 층 2개
- `self.leaky_relu = nn.LeakyReLU(0.2)` : 활성화 함수. 양수는 그대로 두고 음수는 0.2배로 줄입니다.

**계산 과정 (`forward`)**

**① 각 점의 이웃 수(degree) 세기**
- `src, dst = edge_index` : 선의 출발점 목록과 도착점 목록으로 나눕니다.
- `deg = torch.zeros(node_features.size(0), ...)` : 점 개수만큼 0을 준비
- `deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))` : `src`에 **번호가 나올 때마다 그 번호 칸에 1을 더합니다.** 예를 들어 `src = [0, 0, 1]`이면 0번 칸은 2, 1번 칸은 1이 됩니다. 결과적으로 "사용자가 본 영화 수"가 됩니다.
- `dst`에 대해서도 같은 방법으로 "영화를 본 사용자 수"를 셉니다.

**② 정규화 값**
- `norm = 1.0/torch.sqrt(deg[src]*deg[dst])` : 선마다 `1 / √(사용자의 이웃 수 × 영화의 이웃 수)`
- 영화를 수천 편 본 사용자나 수백 명이 본 인기 영화는 메시지가 너무 많이 쌓입니다. **이웃이 많을수록 메시지 하나의 비중을 줄여** 균형을 맞춥니다.

**③ 선마다 메시지 만들기**
- `src_feat = node_features[src]` : 선마다 사용자 쪽 벡터 / `dst_feat = node_features[dst]` : 영화 쪽 벡터

| 코드 | 의미 |
|---|---|
| `self.W1(dst_feat) + self.W2(dst_feat*src_feat)` | **영화 → 사용자**로 가는 메시지 |
| `self.W1(src_feat) + self.W2(src_feat*dst_feat)` | **사용자 → 영화**로 가는 메시지 |
| `... *= norm.unsqueeze(1)` | 정규화 값을 곱함 |

- 메시지는 두 부분의 합입니다. `W1(상대 벡터)`는 **상대의 정보 그 자체**이고, `W2(상대 벡터 * 내 벡터)`는 **나와 상대가 얼마나 닮았는지**를 반영한 정보입니다. `*`는 같은 자리끼리 곱하는 연산입니다.
- `norm.unsqueeze(1)` : `norm`의 모양 `(선 개수,)`을 `(선 개수, 1)`로 바꿔 벡터의 64개 값 모두에 곱해지도록 합니다.

**④ 메시지 모으기**
- `aggregated_messages = torch.zeros_like(node_features)` : 점마다 메시지를 모을 빈 칸
- `aggregated_messages.index_add_(0, dst, edge_messages_for_dst)` : 각 영화 칸에, 그 영화로 들어오는 메시지를 **모두 더합니다.**
- `aggregated_messages[:user_num] += self.W1(node_features[:user_num])` : 사용자 칸에 **자기 자신의 정보**를 더합니다. (`[:user_num]`은 앞쪽 610개 = 사용자)
- `aggregated_messages[user_num:] += self.W1(node_features[user_num:])` : 영화 칸에도 자기 자신의 정보를 더합니다.

**⑤** `self.leaky_relu(aggregated_messages)` : 활성화 함수를 통과시켜 새 벡터를 완성합니다.

> **주의할 줄:** `aggregated_messages.index_add(0, src, edge_messages_for_src)`에는 이름 끝에 `_`가 **없습니다.** PyTorch에서 `_`로 끝나는 함수는 원본을 직접 고치고, `_`가 없는 함수는 **새 결과를 돌려주기만** 합니다. 이 줄은 돌려받은 결과를 어디에도 저장하지 않으므로 **아무 효과가 없습니다.** 즉 이 코드에서는 사용자 쪽으로 가는 메시지가 실제로는 더해지지 않고, 영화 쪽만 이웃의 메시지를 받습니다. 의도대로 하려면 영화 쪽과 똑같이 `index_add_`로 써야 합니다.

In [10]:
# 10
### TODO: NGCF 모델 완성.

class NGCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, layer_dims, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.node_embeddings = nn.Embedding(self.num_users+self.num_items,self.embedding_dim)
        nn.init.xavier_uniform_(self.node_embeddings.weight)

        self.layers = nn.ModuleList([
            NGCFLayer(input_dim=(embedding_dim if i == 0 else layer_dims[i - 1]),
                      output_dim=layer_dims[i], dropout=dropout)
            for i in range(len(layer_dims))
        ])

    def forward(self, edge_index):
        node_features = self.node_embeddings.weight
        layer_outputs = [node_features]
        for layer in self.layers:
            node_features = layer(edge_index, node_features,self.num_users, self.num_items)
            layer_outputs.append(node_features)

        # Hint: NGCF의 final feature(representation)은 layer 별 feature에 대한 concatenated vector
        # Hint: 최종 final feature matrix에는 [feuture_vector for users + feature_vector for items]가 들어있음.

        final_features = torch.concat(layer_outputs,dim=-1)
        user_features = final_features[:self.num_users]
        item_features = final_features[self.num_users:]
        return user_features, item_features

    def bpr_loss(self, user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
        pos_scores = torch.sum(user_emb * pos_item_emb, dim=1)
        neg_scores = torch.sum(user_emb * neg_item_emb, dim=1)
        loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
        reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2) \
                                 + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)
        return loss + reg_loss

### 📘 셀 10 해설
**한 줄 요약:** [실습 정답] 셀 9의 층을 쌓아 **NGCF 모델 전체**를 만들고, 학습에 쓸 **BPR 손실 함수**를 정의합니다.

**부품 (`__init__`)**
- `self.node_embeddings = nn.Embedding(self.num_users+self.num_items, self.embedding_dim)` : **모든 점의 초기 벡터가 담긴 표**입니다. 크기는 10,334줄 × 64칸. 번호를 넣으면 그 줄의 벡터를 꺼내 줍니다. 이 표의 값들이 **학습으로 고쳐지는 대상**입니다.
- `nn.init.xavier_uniform_(...)` : 표의 초기값을 학습하기 좋은 범위의 무작위 값으로 채웁니다.
- `self.layers = nn.ModuleList([... for i in range(len(layer_dims))])` : 셀 9의 `NGCFLayer`를 `layer_dims`의 길이만큼 만들어 담습니다. 셀 14에서 `layer_dims=[64, 64]`를 넣으므로 **2층**입니다.
  - `input_dim=(embedding_dim if i == 0 else layer_dims[i - 1])` : 첫 층의 입력 크기는 임베딩 크기, 그다음 층부터는 **앞 층의 출력 크기**를 씁니다. `A if 조건 else B`는 조건이 참이면 A, 거짓이면 B를 고르는 문법입니다.

**계산 과정 (`forward`)**
- `node_features = self.node_embeddings.weight` : 임베딩 표 전체를 꺼냅니다. `(10334, 64)`
- `layer_outputs = [node_features]` : 초기 벡터를 먼저 보관
- 반복문 : 층을 하나 지날 때마다 결과를 `layer_outputs`에 추가합니다.
- `final_features = torch.concat(layer_outputs, dim=-1)` : 보관한 벡터 3개를 **옆으로 이어 붙입니다.** 64 + 64 + 64 = **192칸**
- `final_features[:self.num_users]` : 앞 610줄 = 사용자 / `[self.num_users:]` : 나머지 = 영화

**왜 층별 결과를 모두 이어 붙일까?**

| 벡터 | 담긴 정보 |
|---|---|
| 초기 벡터 | 자기 자신 |
| 1층 통과 후 | 직접 이어진 이웃 (내가 본 영화) |
| 2층 통과 후 | 이웃의 이웃 (내가 본 영화를 본 다른 사용자들) |

가까운 관계와 먼 관계의 정보를 **모두** 활용하기 위해서입니다.

**`bpr_loss` — 손실 함수**
- `pos_scores = torch.sum(user_emb * pos_item_emb, dim=1)` : 사용자 벡터와 **본 영화** 벡터를 같은 자리끼리 곱해 모두 더한 값(**내적**). 두 벡터가 닮을수록 큽니다. 이 값이 "사용자가 이 영화를 좋아할 점수"입니다.
- `neg_scores` : 같은 방법으로 구한 **안 본 영화**의 점수
- `loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))` : 본 영화의 점수가 안 본 영화의 점수보다 **클수록 손실이 작아집니다.**
- `reg_loss` : 벡터의 값들이 지나치게 커지지 않도록 하는 벌점(정규화). `reg_weight=1e-4`로 아주 작게 반영합니다.

**BPR(Bayesian Personalized Ranking)의 생각:** 점수의 크기 자체를 맞히려는 것이 아니라, **"본 것이 안 본 것보다 위에 오도록" 순서**를 맞히는 것이 목표입니다. 추천은 결국 순위를 매기는 일이기 때문입니다.

# Evaluation Function

In [11]:
# 11
def evaluate(user_features, item_features, test_edge_index, k):
    user_pos_items = defaultdict(list)
    E_test = test_edge_index.size(1)
    for i in range(E_test):
        u = test_edge_index[0, i].item()
        it = test_edge_index[1, i].item() - num_users
        user_pos_items[u].append(it)

    recalls, precisions, ndcgs = [], [], []
    for user, pos_items in user_pos_items.items():
        user_emb = user_features[user]
        scores = torch.matmul(item_features, user_emb)
        topk_scores, topk_indices = torch.topk(scores, k=k)
        topk_indices = topk_indices.cpu().numpy().tolist()

        hits = 0
        dcg = 0.0
        idcg = 0.0
        n_pos = len(pos_items)

        for rank, item_idx in enumerate(topk_indices):
            if item_idx in pos_items:
                hits += 1
                dcg += 1.0 / math.log2(rank + 2)
        for rank in range(min(n_pos, k)):
            idcg += 1.0 / math.log2(rank + 2)

        recall_u = hits / n_pos
        precision_u = hits / k
        ndcg_u = dcg / idcg if idcg > 0 else 0.0

        recalls.append(recall_u)
        precisions.append(precision_u)
        ndcgs.append(ndcg_u)

    recall = np.mean(recalls)
    precision = np.mean(precisions)
    ndcg = np.mean(ndcgs)
    return recall, precision, ndcg

### 📘 셀 11 해설
**한 줄 요약:** 추천 결과를 평가하는 함수입니다. 사용자마다 **점수가 높은 영화 k개**를 뽑아, 숨겨 둔 정답과 얼마나 맞는지 세 가지 지표로 계산합니다.

**① 사용자별 정답 목록 만들기**
- `user_pos_items = defaultdict(list)` : 처음 보는 사용자 번호가 나오면 **자동으로 빈 리스트를 만들어 주는** 사전입니다.
- `test_edge_index[1, i].item() - num_users` : 영화 쪽 번호에서 610을 빼 **원래 영화 번호(0부터)** 로 되돌립니다.
- 결과: `{사용자 번호: [정답 영화 번호들]}`

**② 사용자마다 추천 뽑기**
- `scores = torch.matmul(item_features, user_emb)` : **모든 영화**(9,724편)와 이 사용자의 내적 점수를 한 번에 계산합니다.
- `torch.topk(scores, k=k)` : 점수가 높은 k개의 **(점수, 번호)** 를 돌려줍니다. 번호만 사용합니다.

**③ 맞힌 개수 세기**
- `for rank, item_idx in enumerate(topk_indices):` : 추천 목록을 1등부터 차례로 봅니다(`rank`는 0부터).
- `if item_idx in pos_items:` : 정답 목록에 있으면 `hits`를 1 늘리고 `dcg`에 점수를 더합니다.

**세 가지 지표** (모두 **높을수록 좋음**)

| 지표 | 계산 | 의미 |
|---|---|---|
| **Recall@k** | 맞힌 개수 ÷ 정답 개수 | 정답 중 **몇 %를 찾아냈는가** |
| **Precision@k** | 맞힌 개수 ÷ k | 추천한 것 중 **몇 %가 정답인가** |
| **NDCG@k** | `dcg / idcg` | 정답을 **얼마나 위쪽 순위에** 놓았는가 |

**NDCG 읽기**
- `dcg += 1.0 / math.log2(rank + 2)` : 맞힌 순위가 높을수록 큰 점수를 줍니다. 1등(`rank=0`)에서 맞히면 1 / log₂2 = 1점, 2등이면 약 0.63점, 3등이면 0.5점입니다.
- `idcg` : 정답을 **1등부터 빈틈없이** 맞혔을 때 받을 수 있는 최고 점수
- `ndcg_u = dcg / idcg if idcg > 0 else 0.0` : 최고 점수 대비 비율(0 ~ 1). 같은 개수를 맞혀도 **위쪽에서 맞힐수록** 높습니다.

마지막에 `np.mean(...)`으로 모든 사용자의 평균을 냅니다.

> 이 함수는 사용자가 **학습 데이터에서 이미 본 영화**를 추천 후보에서 빼지 않습니다. 모델은 이미 본 영화에 높은 점수를 주도록 학습되므로 추천 10개의 상당수가 그런 영화로 채워지고, 그만큼 지표가 낮게 나옵니다.

# Train

In [12]:
# 12
def train(model, optimizer, train_edge_index, val_edge_index, num_epochs, batch_size, device, k):
    model.to(device)
    train_edge_index = train_edge_index.to(device)
    val_edge_index = val_edge_index.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        num_batches = len(train_edge_index[0]) // batch_size

        for _ in range(num_batches):
            indices = torch.randint(0, train_edge_index.size(1), (batch_size,), device=device)
            user_indices = train_edge_index[0, indices]
            pos_item_indices = train_edge_index[1, indices] - num_users
            neg_item_indices = torch.randint(0, num_movies, (batch_size,), device=device)

            user_features, item_features = model(train_edge_index)
            u_emb = user_features[user_indices]
            pos_emb = item_features[pos_item_indices]
            neg_emb = item_features[neg_item_indices]

            loss = model.bpr_loss(u_emb, pos_emb, neg_emb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {total_loss / num_batches:.4f}")

        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                user_features, item_features = model(train_edge_index)
                recall, precision, ndcg = evaluate(user_features.cpu(), item_features.cpu(), val_edge_index, k)
                print(f"[Validation] Epoch {epoch + 1}: Recall@{k}: {recall:.4f}, Precision@{k}: {precision:.4f}, NDCG@{k}: {ndcg:.4f}")

### 📘 셀 12 해설
**한 줄 요약:** 모델을 **학습시키는 함수**입니다. 5 에폭마다 검증 데이터로 성능을 확인합니다.

- `num_batches = len(train_edge_index[0]) // batch_size` : 한 에폭에 몇 번 반복할지. `//`는 나눗셈의 **몫**입니다. 학습용 선이 약 8만 개이고 `batch_size=1024`이므로 약 77번입니다.

**묶음 하나를 만드는 과정 (샘플링)**

| 코드 | 의미 |
|---|---|
| `indices = torch.randint(0, 선 개수, (batch_size,))` | 학습용 선 중 1,024개를 **무작위로** 고름 |
| `user_indices = train_edge_index[0, indices]` | 고른 선의 사용자 |
| `pos_item_indices = train_edge_index[1, indices] - num_users` | 고른 선의 영화 = 그 사용자가 **본 영화** (610을 빼 원래 번호로) |
| `neg_item_indices = torch.randint(0, num_movies, (batch_size,))` | 전체 영화 중 **아무거나** 고름 = **안 본 영화**로 간주 |

- 무작위로 고른 영화가 우연히 그 사용자가 본 영화일 수도 있습니다. 하지만 영화가 9,724편이나 되어 그럴 확률이 낮으므로 "안 본 영화"로 취급합니다(negative sampling).

**학습**
- `user_features, item_features = model(train_edge_index)` : **그래프 전체**에 대해 모든 점의 최종 벡터를 계산합니다.
- `u_emb = user_features[user_indices]` 등 : 그중 이번 묶음에 필요한 벡터만 꺼냅니다.
- `loss = model.bpr_loss(u_emb, pos_emb, neg_emb)` : 셀 10의 BPR 손실
- `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` : 기울기 초기화 → 기울기 계산 → 가중치 수정

**검증**
- `if (epoch + 1) % 5 == 0:` : `%`는 **나머지**입니다. 에폭 번호가 5의 배수일 때만 실행합니다.
- `model.eval()`, `with torch.no_grad():` : 평가 모드로 바꾸고 기울기 계산을 끕니다.
- `model(train_edge_index)` : 검증할 때도 그래프는 **학습용 선**으로 만듭니다. 검증용 선은 숨겨 둔 정답으로만 씁니다.
- `evaluate(user_features.cpu(), item_features.cpu(), val_edge_index, k)` : 셀 11의 함수로 평가

# Test

In [13]:
# 13
def test(model, train_edge_index, test_edge_index, k, device):
    model.eval()
    train_edge_index = train_edge_index.to(device)
    test_edge_index = test_edge_index.to(device)

    with torch.no_grad():
        user_features, item_features = model(train_edge_index)

    user_features = user_features.cpu()
    item_features = item_features.cpu()

    recall, precision, ndcg = evaluate(user_features, item_features, test_edge_index, k)
    recall = round(recall, 4)
    precision = round(precision, 4)
    ndcg = round(ndcg, 4)

    print(f"Recall@{k}: {recall:.4f}, Precision@{k}: {precision:.4f}, NDCG@{k}: {ndcg:.4f}")
    return recall, precision, ndcg

### 📘 셀 13 해설
**한 줄 요약:** 학습이 끝난 모델을 **테스트 데이터로 최종 평가**하는 함수입니다.

- 셀 12의 검증 부분과 같은 방식입니다. 그래프는 학습용 선으로 만들고, 정답은 **테스트용 선**을 씁니다.
- `.cpu()` : GPU에 있는 벡터를 CPU로 옮깁니다. `evaluate` 안에서 numpy와 파이썬 반복문을 쓰기 때문입니다.
- `round(recall, 4)` : 소수점 넷째 자리까지 반올림
- `return recall, precision, ndcg` : 세 값을 함께 돌려줍니다.

# Check Model Performace

In [14]:
# 14
ngcf_model = NGCF(num_users, num_movies, embedding_dim=64, layer_dims=[64, 64], dropout=0.1)
optimizer_ngcf = torch.optim.Adam(ngcf_model.parameters(), lr=1e-3)

print("===== Train NGCF =====")
train(
    model=ngcf_model,
    optimizer=optimizer_ngcf,
    train_edge_index=train_edge_index,
    val_edge_index=val_edge_index,
    num_epochs=30,
    batch_size=1024,
    device=device,
    k=10
)

print("===== Test NGCF =====")
test(
    model=ngcf_model,
    train_edge_index=train_edge_index,
    test_edge_index=test_edge_index,
    k=10,
    device=device
)

===== Train NGCF =====
Epoch 1/30, Training Loss: 0.4030
Epoch 2/30, Training Loss: 0.3322
Epoch 3/30, Training Loss: 0.3230
Epoch 4/30, Training Loss: 0.3192
Epoch 5/30, Training Loss: 0.3166
[Validation] Epoch 5: Recall@10: 0.0434, Precision@10: 0.0367, NDCG@10: 0.0496
Epoch 6/30, Training Loss: 0.3083
Epoch 7/30, Training Loss: 0.3083
Epoch 8/30, Training Loss: 0.2976
Epoch 9/30, Training Loss: 0.2867
Epoch 10/30, Training Loss: 0.2769
[Validation] Epoch 10: Recall@10: 0.0366, Precision@10: 0.0328, NDCG@10: 0.0403
Epoch 11/30, Training Loss: 0.2663
Epoch 12/30, Training Loss: 0.2503
Epoch 13/30, Training Loss: 0.2365
Epoch 14/30, Training Loss: 0.2246
Epoch 15/30, Training Loss: 0.2151
[Validation] Epoch 15: Recall@10: 0.0392, Precision@10: 0.0338, NDCG@10: 0.0433
Epoch 16/30, Training Loss: 0.2099
Epoch 17/30, Training Loss: 0.2033
Epoch 18/30, Training Loss: 0.2023
Epoch 19/30, Training Loss: 0.1958
Epoch 20/30, Training Loss: 0.1942
[Validation] Epoch 20: Recall@10: 0.0513, Preci

(np.float64(0.0577), np.float64(0.0416), np.float64(0.0584))

### 📘 셀 14 해설
**한 줄 요약:** NGCF 모델을 만들고, **30 에폭 학습**한 뒤 **테스트**합니다.

- `NGCF(num_users, num_movies, embedding_dim=64, layer_dims=[64, 64], dropout=0.1)` : 임베딩 64칸, 층 2개인 모델
- `torch.optim.Adam(ngcf_model.parameters(), lr=1e-3)` : 옵티마이저. 학습률 0.001
- `train(model=ngcf_model, ..., k=10)` : 인자를 `이름=값` 형태로 넘기는 **키워드 인자**입니다. 순서를 헷갈릴 일이 없고 읽기 쉽습니다.
- `k=10` : 사용자마다 10개를 추천해 평가합니다.

**출력 해석**
- **학습 손실:** `0.4030 → 0.1656` 으로 꾸준히 감소. "본 영화의 점수 > 안 본 영화의 점수"가 되도록 학습이 진행되고 있습니다.
- **검증 성능 (Recall@10):** `0.0434 → 0.0366 → 0.0392 → 0.0513 → 0.0509 → 0.0538`. 손실처럼 매끄럽게 좋아지지 않고 오르내리지만 전체적으로는 올라갑니다.
- **테스트 성능:** `Recall@10: 0.0577`, `Precision@10: 0.0416`, `NDCG@10: 0.0584`
  - Recall 0.0577 : 숨겨 둔 정답 영화 중 **약 5.8%** 가 추천 10개 안에 들어왔습니다.
  - Precision 0.0416 : 추천 10개 중 평균 **0.4개**가 정답입니다.
- 맨 끝의 `(np.float64(0.0577), ...)` : 셀의 마지막 줄이 `test(...)` 호출이어서, 그 함수가 **돌려준 값**을 주피터가 자동으로 보여 준 것입니다.

**숫자가 낮아 보이는 이유**
1. 9,724편 중에서 10편만 고르는 문제라 원래 어렵습니다(무작위로 고르면 훨씬 낮습니다).
2. 이미 본 영화를 추천 후보에서 빼지 않았습니다(셀 11 해설).
3. 셀 9의 `index_add` 줄 때문에 사용자 쪽은 이웃의 메시지를 받지 못합니다(셀 9 해설).
4. 30 에폭은 짧은 편이고, 손실이 아직 줄어드는 중입니다.

**전체 실습 정리**
1. 사용자–영화 평점 기록은 **그래프**로 볼 수 있고, 선 목록(`edge_index`)으로 간단히 표현합니다.
2. NGCF는 각 점이 **이웃의 벡터를 모아** 자기 벡터를 갱신하는 층을 쌓아, "같은 영화를 본 사람들"처럼 **한 다리 건너의 관계**까지 벡터에 담습니다.
3. 사용자와 영화의 벡터를 **내적**한 값이 추천 점수입니다.
4. **BPR 손실**은 점수의 크기가 아니라 "본 것이 안 본 것보다 위"라는 **순서**를 학습합니다.
5. 평가는 추천 상위 k개를 정답과 비교하는 **Recall@k, Precision@k, NDCG@k**로 합니다.